In [4]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
import sys, os

from pathlib import Path

DATA_DIR = Path("rounds/1_round/data")
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../../..")))


In [5]:
files = [
    "prices_round_1_day_-1.csv",
    "prices_round_1_day_-2.csv",
    "prices_round_1_day_0.csv"
]

dfs = [pd.read_csv(f, sep=';') for f in files]
full_prices = pd.concat(dfs, ignore_index=True)

full_prices.head()


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
0,-1,0,INTARIAN_PEPPER_ROOT,10991.0,15.0,NaN,NaN,NaN,NaN,11006.0,10.0,11009.0,15.0,NaN,NaN,10998.5,0.0
1,-1,0,ASH_COATED_OSMIUM,NaN,NaN,NaN,NaN,NaN,NaN,10003.0,29.0,NaN,NaN,NaN,NaN,10003.0,0.0
2,-1,100,ASH_COATED_OSMIUM,9984.0,11.0,NaN,NaN,NaN,NaN,10000.0,11.0,10003.0,22.0,NaN,NaN,9992.0,0.0
3,-1,100,INTARIAN_PEPPER_ROOT,10994.0,9.0,10991.0,21.0,NaN,NaN,11006.0,9.0,11009.0,21.0,NaN,NaN,11000.0,0.0
4,-1,200,ASH_COATED_OSMIUM,9985.0,15.0,9982.0,20.0,NaN,NaN,10001.0,15.0,10003.0,20.0,NaN,NaN,9993.0,0.0


In [6]:
#ash = full_prices[full_prices["product"] == "ASH_COATED_OSMIUM"].copy()
#ash = ash.sort_values("timestamp").reset_index(drop=True)

#ash.head()


price_df = pd.read_csv("prices_round_1_day_-1.csv" , sep = ';')

#price_df.head(5)
pd.set_option("display.max_rows", 200)

ash_df = price_df[price_df['product']=='ASH_COATED_OSMIUM'].reset_index(drop=True)
ash_df.iloc[100:200]


,day,timestamp,product,bid_price_1,bid_volume_1,bid_price_2,bid_volume_2,bid_price_3,bid_volume_3,ask_price_1,ask_volume_1,ask_price_2,ask_volume_2,ask_price_3,ask_volume_3,mid_price,profit_and_loss
100,-1,10000,ASH_COATED_OSMIUM,9988.0,14.0,9986.0,28.0,NaN,NaN,10007.0,28.0,NaN,NaN,NaN,NaN,9997.5,0.0
101,-1,10100,ASH_COATED_OSMIUM,9988.0,15.0,9986.0,24.0,NaN,NaN,10004.0,15.0,NaN,NaN,NaN,NaN,9996.0,0.0
102,-1,10200,ASH_COATED_OSMIUM,9988.0,13.0,9986.0,26.0,NaN,NaN,10004.0,13.0,10007.0,26.0,NaN,NaN,9996.0,0.0
103,-1,10300,ASH_COATED_OSMIUM,9988.0,13.0,NaN,NaN,NaN,NaN,10004.0,13.0,10007.0,27.0,NaN,NaN,9996.0,0.0
104,-1,10400,ASH_COATED_OSMIUM,9986.0,28.0,NaN,NaN,NaN,NaN,10005.0,14.0,10007.0,28.0,NaN,NaN,9995.5,0.0
105,-1,10500,ASH_COATED_OSMIUM,9988.0,15.0,9986.0,29.0,NaN,NaN,10004.0,15.0,10007.0,29.0,NaN,NaN,9996.0,0.0
106,-1,10600,ASH_COATED_OSMIUM,9989.0,15.0,9987.0,29.0,NaN,NaN,10005.0,15.0,10008.0,29.0,NaN,NaN,9997.0,0.0
107,-1,10700,ASH_COATED_OSMIUM,9989.0,10.0,9986.0,28.0,NaN,NaN,10005.0,10.0,10007.0,28.0,NaN,NaN,9997.0,0.0
108,-1,10800,ASH_COATED_OSMIUM,9986.0,22.0,NaN,NaN,NaN,NaN,10004.0,15.0,10007.0,22.0,NaN,NaN,9995.0,0.0
109,-1,10900,ASH_COATED_OSMIUM,9986.0,27.0,NaN,NaN,NaN,NaN,10004.0,15.0,10007.0,27.0,NaN,NaN,9995.0,0.0


In [7]:
# ══════════════════════════════════════════════════════════════════════════════
# COMPREHENSIVE MEAN-REVERSION / STATIONARITY SUITE — ASH_COATED_OSMIUM
# Tests: ADF, KPSS, Phillips-Perron, Hurst exponent, OU half-life,
#        Variance-Ratio, ACF/PACF diagnostics
# ══════════════════════════════════════════════════════════════════════════════
import warnings, matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from statsmodels.tsa.stattools import adfuller, kpss, acf, pacf
from statsmodels.tsa.stattools import adfuller as _adf
from statsmodels.regression.linear_model import OLS
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
import statsmodels.api as sm

warnings.filterwarnings("ignore", category=FutureWarning)

# ── helpers ───────────────────────────────────────────────────────────────────
def _clean(x):
    x = pd.to_numeric(x, errors="coerce").replace([np.inf, -np.inf], np.nan).dropna()
    return x

def _logret(x):
    x = _clean(x); x = x[x > 0]
    return np.log(x).diff().replace([np.inf, -np.inf], np.nan).dropna()

# ── build series ──────────────────────────────────────────────────────────────
ash = full_prices[full_prices["product"] == "ASH_COATED_OSMIUM"].copy()
ash = ash.sort_values(["day", "timestamp"]).reset_index(drop=True)
if "mid_price" in ash.columns and ash["mid_price"].notna().any():
    ash["price"] = pd.to_numeric(ash["mid_price"], errors="coerce")
else:
    ash["price"] = (
        pd.to_numeric(ash["bid_price_1"], errors="coerce")
        + pd.to_numeric(ash["ask_price_1"], errors="coerce")
    ) / 2.0

p_all = _clean(ash["price"]).values
lr_all = _logret(ash["price"]).values

# ══════════════════════════════════════════════════════════════════════════════
# 1. UNIT ROOT / STATIONARITY TESTS  (per day + pooled)
# ══════════════════════════════════════════════════════════════════════════════
def _pp(x):
    res = adfuller(x, maxlag=0, regression="c", autolag=None)
    return res[0], res[1]

rows = []
groups = list(ash.groupby("day", sort=True)) + [("ALL", ash)]
for d, g in groups:
    p = _clean(g["price"]); lr = _logret(g["price"])
    if len(p) < 50 or len(lr) < 50:
        continue
    adf_l = adfuller(p, regression="c", autolag="AIC")
    kpss_l = kpss(p, regression="c", nlags="auto")
    pp_l  = _pp(p)
    adf_r = adfuller(lr, regression="c", autolag="AIC")
    kpss_r = kpss(lr, regression="c", nlags="auto")
    pp_r  = _pp(lr)
    rows.append({
        "day": int(d) if d != "ALL" else "ALL", "n": int(len(p)),
        "ADF_p (level)": round(adf_l[1], 4),
        "KPSS_p (level)": f">={kpss_l[1]:.2f}" if kpss_l[1] >= 0.1 else round(kpss_l[1], 4),
        "PP_p (level)": round(pp_l[1], 4),
        "ADF_p (logret)": round(adf_r[1], 4),
        "KPSS_p (logret)": f">={kpss_r[1]:.2f}" if kpss_r[1] >= 0.1 else round(kpss_r[1], 4),
        "PP_p (logret)": round(pp_r[1], 4),
    })
unit_root_df = pd.DataFrame(rows)
print("=" * 80)
print("1. UNIT-ROOT / STATIONARITY TESTS")
print("   ADF/PP: p<0.05 -> reject unit root (stationary)")
print("   KPSS:   p>0.05 -> fail to reject stationarity (stationary)")
print("=" * 80)
display(unit_root_df)

# ══════════════════════════════════════════════════════════════════════════════
# 2. HURST EXPONENT  (rescaled range)
# ══════════════════════════════════════════════════════════════════════════════
def hurst_rs(ts, min_window=20):
    ts = np.asarray(ts, float); n = len(ts)
    max_k = int(np.floor(n / min_window))
    if max_k < 2: return np.nan
    ns, rs_vals = [], []
    for k in range(2, max_k + 1):
        size = n // k
        if size < min_window: break
        rs_list = []
        for i in range(k):
            chunk = ts[i*size:(i+1)*size]
            devs = np.cumsum(chunk - chunk.mean())
            R = devs.max() - devs.min(); S = chunk.std(ddof=1)
            if S > 0: rs_list.append(R / S)
        if rs_list:
            ns.append(size); rs_vals.append(np.mean(rs_list))
    if len(ns) < 3: return np.nan
    return np.polyfit(np.log(ns), np.log(rs_vals), 1)[0]

hurst_level  = hurst_rs(p_all)
hurst_logret = hurst_rs(lr_all)
print("\n" + "=" * 80)
print("2. HURST EXPONENT (rescaled range)")
print("   H < 0.5 -> mean-reverting | H = 0.5 -> random walk | H > 0.5 -> trending")
print("=" * 80)
print(f"   Price level:  H = {hurst_level:.4f}  {'<- mean-reverting' if hurst_level < 0.5 else '<- trending' if hurst_level > 0.5 else '<- random walk'}")
print(f"   Log returns:  H = {hurst_logret:.4f}  {'<- mean-reverting' if hurst_logret < 0.5 else '<- trending' if hurst_logret > 0.5 else '<- random walk'}")

# ══════════════════════════════════════════════════════════════════════════════
# 3. ORNSTEIN-UHLENBECK HALF-LIFE
# ══════════════════════════════════════════════════════════════════════════════
def ou_half_life(prices):
    y = np.asarray(prices, float); dy = np.diff(y); y_lag = y[:-1]
    res = OLS(dy, sm.add_constant(y_lag)).fit()
    theta = -res.params[1]
    if theta <= 0: return np.inf, theta, res.pvalues[1]
    return np.log(2) / theta, theta, res.pvalues[1]

hl, theta, theta_p = ou_half_life(p_all)
print("\n" + "=" * 80)
print("3. OU HALF-LIFE OF MEAN REVERSION (price level)")
print("   Regresses dp on p_{t-1}; theta>0 -> mean-reverting")
print("=" * 80)
print(f"   theta (speed) = {theta:.6f}  (p-value = {theta_p:.4e})")
print(f"   Half-life     = {hl:.1f} ticks")
if hl < np.inf:
    print(f"   Price reverts halfway to mean in ~{hl:.0f} ticks")
else:
    print("   No mean-reversion detected (theta <= 0)")

# ══════════════════════════════════════════════════════════════════════════════
# 4. VARIANCE RATIO TEST
# ══════════════════════════════════════════════════════════════════════════════
def variance_ratio(x, lags=None):
    if lags is None: lags = [2, 4, 8, 16, 32, 64]
    x = np.asarray(x, float); var1 = np.var(np.diff(x), ddof=1)
    results = []
    for q in lags:
        diffs = x[q:] - x[:-q]
        vr = (np.var(diffs, ddof=1) / q) / var1 if var1 > 0 else np.nan
        results.append({"lag": q, "VR": round(vr, 4)})
    return results

vr_results = variance_ratio(p_all)
vr_df = pd.DataFrame(vr_results)
print("\n" + "=" * 80)
print("4. VARIANCE RATIO (price level)")
print("   VR < 1 -> mean-reverting | VR = 1 -> random walk | VR > 1 -> trending")
print("=" * 80)
display(vr_df)


C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\2223329802.py:53: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_l = kpss(p, regression="c", nlags="auto")
C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\2223329802.py:56: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_r = kpss(lr, regression="c", nlags="auto")
C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\2223329802.py:53: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_l = kpss(p, regression="c", nlags="auto")
C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\2223329802.py:56: InterpolationWarning: The test statistic is outside of the range of p-values av

1. UNIT-ROOT / STATIONARITY TESTS
   ADF/PP: p<0.05 -> reject unit root (stationary)
   KPSS:   p>0.05 -> fail to reject stationarity (stationary)


C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\2223329802.py:56: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_r = kpss(lr, regression="c", nlags="auto")


,day,n,ADF_p (level),KPSS_p (level),PP_p (level),ADF_p (logret),KPSS_p (logret),PP_p (logret)
0,-2,10000,0.0,>=0.10,0.0,0.0,>=0.10,0.0
1,-1,10000,0.0,>=0.10,0.0,0.0,>=0.10,0.0
2,0,10000,0.0,>=0.10,0.0,0.0,>=0.10,0.0
3,ALL,30000,0.0,>=0.10,0.0,0.0,>=0.10,0.0



2. HURST EXPONENT (rescaled range)
   H < 0.5 -> mean-reverting | H = 0.5 -> random walk | H > 0.5 -> trending
   Price level:  H = 0.7427  <- trending
   Log returns:  H = 0.2100  <- mean-reverting

3. OU HALF-LIFE OF MEAN REVERSION (price level)
   Regresses dp on p_{t-1}; theta>0 -> mean-reverting
   theta (speed) = 1.001741  (p-value = 0.0000e+00)
   Half-life     = 0.7 ticks
   Price reverts halfway to mean in ~1 ticks

4. VARIANCE RATIO (price level)
   VR < 1 -> mean-reverting | VR = 1 -> random walk | VR > 1 -> trending


,lag,VR
0,2,0.4999
1,4,0.2500
2,8,0.1250
3,16,0.0625
4,32,0.0313
5,64,0.0157


In [15]:
# =============================================================================
# PLOTLY VISUALISATIONS - MEAN-REVERSION DIAGNOSTICS (run cell 3 first)
# =============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots

NLAGS_ACF = 100
MID_MA_WINDOW = 200
PRICE_SCALE = 100.0   # display price in smaller denomination
RET_SCALE = 10000.0   # display returns in bps

# Clean plotting series: replace literal 0 with NA so Plotly doesn't draw hard drops.
p_plot = pd.Series(p_all, dtype=float).replace(0.0, np.nan)
lr_plot = pd.Series(lr_all, dtype=float).replace(0.0, np.nan)

# Keep clean no-NA versions for statistical transforms.
p_stat = p_plot.dropna().values
lr_stat = lr_plot.dropna().values

plot_df = pd.DataFrame({
    "tick": np.arange(len(p_plot)),
    "mid_price": p_plot,
})
plot_df["mid_price_ma"] = plot_df["mid_price"].rolling(MID_MA_WINDOW, min_periods=MID_MA_WINDOW // 2).mean()
plot_df["mid_price_scaled"] = plot_df["mid_price"] / PRICE_SCALE
plot_df["mid_price_ma_scaled"] = plot_df["mid_price_ma"] / PRICE_SCALE

ret_df = pd.DataFrame({
    "tick": np.arange(len(lr_plot)),
    "log_ret": lr_plot,
})
ret_df["log_ret_bps"] = ret_df["log_ret"] * RET_SCALE
ret_df["roll_std_200_bps"] = ret_df["log_ret_bps"].rolling(200, min_periods=100).std()

# --- FIG 1: Price, MA, log returns, rolling vol --------------------------------
fig1 = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.06,
    row_heights=[0.45, 0.27, 0.28],
    subplot_titles=(
        f"ASH_COATED_OSMIUM Mid Price (OU half-life ~ {hl:.0f} ticks)" if hl < np.inf else "ASH_COATED_OSMIUM Mid Price",
        "Log Returns",
        "Rolling Volatility (200 ticks)",
    ),
)

fig1.add_trace(
    go.Scattergl(
        x=plot_df["tick"], y=plot_df["mid_price_scaled"], mode="lines", name=f"Mid Price (/{PRICE_SCALE:.0f})",
        line=dict(color="steelblue", width=1), connectgaps=False,
    ),
    row=1, col=1,
)
fig1.add_trace(
    go.Scattergl(
        x=plot_df["tick"], y=plot_df["mid_price_ma_scaled"], mode="lines", name=f"MA({MID_MA_WINDOW}) (/{PRICE_SCALE:.0f})",
        line=dict(color="orange", width=2), connectgaps=False,
    ),
    row=1, col=1,
)

fig1.add_trace(
    go.Scattergl(
        x=ret_df["tick"], y=ret_df["log_ret_bps"], mode="lines", name="Log Return (bps)",
        line=dict(color="gray", width=1), connectgaps=False,
    ),
    row=2, col=1,
)
fig1.add_hline(y=0, line_width=1, line_dash="dash", line_color="black", row=2, col=1)

fig1.add_trace(
    go.Scattergl(
        x=ret_df["tick"], y=ret_df["roll_std_200_bps"], mode="lines", name="Rolling Std(200) bps",
        line=dict(color="crimson", width=1.5), connectgaps=False,
    ),
    row=3, col=1,
)

fig1.update_layout(height=900, width=1200, title="Mean-Reversion Core Diagnostics", hovermode="x unified", template="plotly_white")
fig1.update_yaxes(title_text=f"Mid Price (/{PRICE_SCALE:.0f})", row=1, col=1)
fig1.update_yaxes(title_text="Log Return (bps)", row=2, col=1)
fig1.update_yaxes(title_text="Vol (bps)", row=3, col=1)
fig1.update_xaxes(title_text="Tick", row=3, col=1)
fig1.show()

# --- FIG 2: ACF/PACF as Plotly bar charts -------------------------------------
acf_p = acf(p_stat, nlags=NLAGS_ACF, fft=True)
pacf_p = pacf(p_stat, nlags=NLAGS_ACF, method="ywm")
acf_r = acf(lr_stat, nlags=NLAGS_ACF, fft=True)
pacf_r = pacf(lr_stat, nlags=NLAGS_ACF, method="ywm")
lags = np.arange(len(acf_p))

fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=("ACF - Price Level", "PACF - Price Level", "ACF - Log Returns", "PACF - Log Returns"),
    vertical_spacing=0.12,
)
fig2.add_trace(go.Bar(x=lags, y=acf_p, name="ACF Price", marker_color="royalblue", showlegend=False), row=1, col=1)
fig2.add_trace(go.Bar(x=lags, y=pacf_p, name="PACF Price", marker_color="mediumpurple", showlegend=False), row=1, col=2)
fig2.add_trace(go.Bar(x=lags, y=acf_r, name="ACF Returns", marker_color="teal", showlegend=False), row=2, col=1)
fig2.add_trace(go.Bar(x=lags, y=pacf_r, name="PACF Returns", marker_color="indianred", showlegend=False), row=2, col=2)
fig2.update_layout(height=800, width=1200, title="Autocorrelation Diagnostics")
fig2.update_xaxes(title_text="Lag")
fig2.update_yaxes(title_text="Correlation")
fig2.show()

# --- FIG 3: Variance Ratio ----------------------------------------------------
vr_lags = [r["lag"] for r in vr_results]
vr_vals = [r["VR"] for r in vr_results]
fig3 = go.Figure()
fig3.add_trace(go.Bar(x=vr_lags, y=vr_vals, name="Variance Ratio", marker_color="teal"))
fig3.add_hline(y=1.0, line_dash="dash", line_color="red", annotation_text="Random Walk (VR=1)")
fig3.update_layout(title="Variance Ratio Test - Price Level", xaxis_title="Lag (ticks)", yaxis_title="VR", height=450, width=950)
fig3.show()

# --- FIG 4: Rolling ADF p-value ----------------------------------------------
ROLL_WIN = 2000
ROLL_STEP = 500
adf_roll_idx, adf_roll_p = [], []
for start in range(0, len(p_stat) - ROLL_WIN, ROLL_STEP):
    chunk = p_stat[start:start + ROLL_WIN]
    try:
        res = _adf(chunk, regression="c", autolag="AIC")
        adf_roll_idx.append(start + ROLL_WIN // 2)
        adf_roll_p.append(res[1])
    except Exception:
        pass

fig4 = go.Figure()
fig4.add_trace(go.Scatter(x=adf_roll_idx, y=adf_roll_p, mode="lines+markers", name="ADF p-value", line=dict(color="navy")))
fig4.add_hline(y=0.05, line_dash="dash", line_color="red", annotation_text="p=0.05")
fig4.update_layout(
    title=f"Rolling ADF p-value (window={ROLL_WIN}, step={ROLL_STEP})",
    xaxis_title="Center tick",
    yaxis_title="ADF p-value",
    height=420,
    width=1100,
)
fig4.show()

# --- FIG 5: Hurst breakdown ---------------------------------------------------
hurst_rows = []
for d, g in ash.groupby("day", sort=True):
    p = _clean(g["price"]).values
    lr = _logret(g["price"]).values
    if len(p) > 200:
        hurst_rows.append({"day": str(int(d)), "H_level": hurst_rs(p), "H_logret": hurst_rs(lr)})
hurst_rows.append({"day": "ALL", "H_level": hurst_level, "H_logret": hurst_logret})
hurst_df = pd.DataFrame(hurst_rows)

fig5 = make_subplots(rows=1, cols=2, subplot_titles=("Hurst - Price Level", "Hurst - Log Returns"))
fig5.add_trace(go.Bar(x=hurst_df["day"], y=hurst_df["H_level"], name="H Level", marker_color="mediumpurple"), row=1, col=1)
fig5.add_trace(go.Bar(x=hurst_df["day"], y=hurst_df["H_logret"], name="H LogRet", marker_color="salmon"), row=1, col=2)
fig5.add_hline(y=0.5, line_dash="dash", line_color="red", row=1, col=1)
fig5.add_hline(y=0.5, line_dash="dash", line_color="red", row=1, col=2)
fig5.update_layout(title="Hurst Exponent Breakdown", height=500, width=1100)
fig5.show()

# --- FIG 6: Plotly summary table ---------------------------------------------
def _verdict(test, val):
    if test == "ADF":
        return ("STATIONARY", "green") if val < 0.05 else ("NON-STATIONARY", "red")
    if test == "KPSS":
        return ("STATIONARY", "green") if val > 0.05 else ("NON-STATIONARY", "red")
    if test == "Hurst":
        if val < 0.45:
            return "MEAN-REVERTING", "green"
        if val > 0.55:
            return "TRENDING", "red"
        return "RANDOM WALK", "orange"
    if test == "VR_avg":
        if val < 0.95:
            return "MEAN-REVERTING", "green"
        if val > 1.05:
            return "TRENDING", "red"
        return "RANDOM WALK", "orange"
    if test == "OU_hl":
        return (f"HL={val:.0f}t", "green") if val < np.inf and val > 0 else ("NO REVERSION", "red")
    return "?", "gray"

kpss_level = float(str(unit_root_df[unit_root_df["day"] == "ALL"]["KPSS_p (level)"].values[0]).replace(">=", ""))
kpss_logret = float(str(unit_root_df[unit_root_df["day"] == "ALL"]["KPSS_p (logret)"].values[0]).replace(">=", ""))

summary_items = [
    ("ADF (level)", "ADF", float(unit_root_df[unit_root_df["day"] == "ALL"]["ADF_p (level)"].values[0])),
    ("ADF (logret)", "ADF", float(unit_root_df[unit_root_df["day"] == "ALL"]["ADF_p (logret)"].values[0])),
    ("KPSS (level)", "KPSS", kpss_level),
    ("KPSS (logret)", "KPSS", kpss_logret),
    ("Hurst (level)", "Hurst", float(hurst_level)),
    ("Hurst (logret)", "Hurst", float(hurst_logret)),
    ("Variance Ratio (avg)", "VR_avg", float(np.mean(vr_vals))),
    ("OU Half-Life", "OU_hl", float(hl) if hl < np.inf else hl),
]

labels = []
values = []
verdicts = []
verdict_colors = []
for label, test, value in summary_items:
    vtxt, vcol = _verdict(test, value)
    labels.append(label)
    values.append(f"{value:.4f}" if np.isfinite(value) else "inf")
    verdicts.append(vtxt)
    verdict_colors.append(vcol)

fig6 = go.Figure(data=[go.Table(
    header=dict(values=["Metric", "Value", "Verdict"], fill_color="#1f2937", font=dict(color="white", size=13), align="left"),
    cells=dict(
        values=[labels, values, verdicts],
        fill_color=[ ["#f9fafb"] * len(labels), ["#f9fafb"] * len(labels), ["#f0fdf4" if c=="green" else "#fff7ed" if c=="orange" else "#fef2f2" for c in verdict_colors] ],
        align="left",
        height=30,
        font=dict(color=[ ["#111827"] * len(labels), ["#111827"] * len(labels), verdict_colors ], size=12),
    )
)])
fig6.update_layout(title="MEAN-REVERSION VERDICT DASHBOARD - ASH_COATED_OSMIUM", height=450, width=980)
fig6.show()

print("All Plotly diagnostics complete.")




KeyError: 'price'

In [ ]:
from order_flow_analysis import detect_levels, level_coverage

detect_levels(ash_df)
level_coverage(ash_df)



Detected 3 order book levels

  Level       Bid %    Ask %     Use?
  ------------------------------------
  L1          96.0%    96.1%        ✅
  L2          65.0%    65.3%        ✅
  L3           2.4%     2.4%        ❌

  Recommended n_levels: 2


({1: (np.float64(96.02000000000001), np.float64(96.06), np.True_),
  2: (np.float64(65.03999999999999), np.float64(65.29), np.True_),
  3: (np.float64(2.4), np.float64(2.44), np.False_)},
 2)

In [ ]:
# =============================================================================
# KALMAN FAIR VALUE + MOMENTUM (uses trader params from kalman_market_making.py)
# =============================================================================
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.stattools import adfuller, kpss

# Parameters from Rounds/0_tutorial/traders/kalman_market_making.py
KF_R_OBS = 1.0
KF_Q_LEVEL = 0.3
KF_Q_DRIFT = 0.015
KF_DRIFT_EPS = 0.04

PRICE_SCALE = 100.0

# Use mid-price series from earlier cell, treat 0 as missing.
price_series = pd.Series(p_all, dtype=float).replace(0.0, np.nan).dropna().values


def kalman_local_trend(prices, r_obs, q_level, q_drift):
    """2D local linear trend Kalman filter.
    state x_t = [mu_t, beta_t], obs z_t = mu_t + noise
    """
    F = np.array([[1.0, 1.0], [0.0, 1.0]])
    H = np.array([[1.0, 0.0]])
    Q = np.array([[q_level, 0.0], [0.0, q_drift]])
    R = np.array([[r_obs]])

    x = np.array([prices[0], 0.0], dtype=float)
    P = np.eye(2) * 10.0

    mu_hist = np.zeros(len(prices))
    beta_hist = np.zeros(len(prices))

    for i, z in enumerate(prices):
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q

        y = np.array([z]) - (H @ x_pred)
        S = H @ P_pred @ H.T + R
        K = P_pred @ H.T @ np.linalg.inv(S)

        x = x_pred + (K @ y)
        P = (np.eye(2) - K @ H) @ P_pred

        mu_hist[i] = x[0]
        beta_hist[i] = x[1]

    resid = prices - mu_hist
    return mu_hist, beta_hist, resid


mu, beta, resid = kalman_local_trend(
    price_series,
    r_obs=KF_R_OBS,
    q_level=KF_Q_LEVEL,
    q_drift=KF_Q_DRIFT,
)

# Momentum regime labels using same epsilon convention as trader.
buy_signal = beta > KF_DRIFT_EPS
sell_signal = beta < -KF_DRIFT_EPS
flat_signal = ~(buy_signal | sell_signal)

fair_df = pd.DataFrame({
    "tick": np.arange(len(price_series)),
    "price": price_series,
    "fair_value": mu,
    "beta": beta,
    "residual": resid,
    "buy_signal": buy_signal,
    "sell_signal": sell_signal,
    "flat_signal": flat_signal,
})

fair_df["price_scaled"] = fair_df["price"] / PRICE_SCALE
fair_df["fv_scaled"] = fair_df["fair_value"] / PRICE_SCALE
fair_df["residual_z"] = (
    (fair_df["residual"] - fair_df["residual"].rolling(500, min_periods=250).mean())
    / fair_df["residual"].rolling(500, min_periods=250).std()
)

# Quick diagnostics for whether fair value appears moving with momentum.
h_mu = hurst_rs(pd.Series(mu).dropna().values)
h_beta = hurst_rs(pd.Series(beta).dropna().values)
h_resid = hurst_rs(pd.Series(resid).dropna().values)

adf_resid_p = adfuller(pd.Series(resid).dropna().values, regression="c", autolag="AIC")[1]
kpss_resid_p = kpss(pd.Series(resid).dropna().values, regression="c", nlags="auto")[1]

signal_share = pd.Series(np.where(buy_signal, "BUY", np.where(sell_signal, "SELL", "FLAT"))).value_counts(normalize=True)

print("Kalman params in use:")
print(f"  r_obs={KF_R_OBS}, q_level={KF_Q_LEVEL}, q_drift={KF_Q_DRIFT}, drift_eps={KF_DRIFT_EPS}")
print("\nMean-reversion / momentum diagnostics:")
print(f"  H(mu)={h_mu:.3f}  | H(beta)={h_beta:.3f} | H(residual)={h_resid:.3f}")
print(f"  Residual ADF p={adf_resid_p:.4g} | KPSS p={kpss_resid_p:.4g}")
print("\nSignal share:")
display(signal_share.rename("share"))

# Plotly dashboard: price vs fair value + beta + residual/zscore
fig = make_subplots(
    rows=3,
    cols=1,
    shared_xaxes=True,
    vertical_spacing=0.05,
    row_heights=[0.5, 0.22, 0.28],
    subplot_titles=(
        "Price vs Kalman Fair Value",
        "Momentum State (beta)",
        "Residual and Residual Z-score",
    ),
)

fig.add_trace(
    go.Scattergl(x=fair_df["tick"], y=fair_df["price_scaled"], mode="lines", name="Price/100", line=dict(color="steelblue", width=1)),
    row=1, col=1,
)
fig.add_trace(
    go.Scattergl(x=fair_df["tick"], y=fair_df["fv_scaled"], mode="lines", name="Fair Value/100", line=dict(color="orange", width=2)),
    row=1, col=1,
)

# Highlight buy/sell momentum regions with markers.
buy_pts = fair_df[fair_df["buy_signal"]]
sell_pts = fair_df[fair_df["sell_signal"]]
fig.add_trace(
    go.Scattergl(x=buy_pts["tick"], y=buy_pts["fv_scaled"], mode="markers", name="BUY regime", marker=dict(color="green", size=4, opacity=0.6)),
    row=1, col=1,
)
fig.add_trace(
    go.Scattergl(x=sell_pts["tick"], y=sell_pts["fv_scaled"], mode="markers", name="SELL regime", marker=dict(color="red", size=4, opacity=0.6)),
    row=1, col=1,
)

fig.add_trace(
    go.Scattergl(x=fair_df["tick"], y=fair_df["beta"], mode="lines", name="beta", line=dict(color="purple", width=1)),
    row=2, col=1,
)
fig.add_hline(y=KF_DRIFT_EPS, line_dash="dash", line_color="green", row=2, col=1)
fig.add_hline(y=-KF_DRIFT_EPS, line_dash="dash", line_color="red", row=2, col=1)
fig.add_hline(y=0.0, line_dash="dot", line_color="black", row=2, col=1)

fig.add_trace(
    go.Scattergl(x=fair_df["tick"], y=fair_df["residual"], mode="lines", name="residual", line=dict(color="gray", width=1)),
    row=3, col=1,
)
fig.add_trace(
    go.Scattergl(x=fair_df["tick"], y=fair_df["residual_z"], mode="lines", name="residual z", line=dict(color="teal", width=1)),
    row=3, col=1,
)
fig.add_hline(y=0.0, line_dash="dot", line_color="black", row=3, col=1)
fig.add_hline(y=2.0, line_dash="dash", line_color="orange", row=3, col=1)
fig.add_hline(y=-2.0, line_dash="dash", line_color="orange", row=3, col=1)

fig.update_layout(
    title=(
        "Kalman Fair Value + Momentum (Trader Params) | "
        f"H(mu)={h_mu:.3f}, H(beta)={h_beta:.3f}, H(resid)={h_resid:.3f}"
    ),
    width=1200,
    height=900,
    hovermode="x unified",
    template="plotly_white",
)
fig.update_yaxes(title_text="Price / 100", row=1, col=1)
fig.update_yaxes(title_text="beta", row=2, col=1)
fig.update_yaxes(title_text="residual / z", row=3, col=1)
fig.update_xaxes(title_text="Tick", row=3, col=1)
fig.show()

display(fair_df.head())
print("Done: using fixed kalman_market_making params to visualize moving fair value + momentum.")

Kalman params in use:
  r_obs=1.0, q_level=0.3, q_drift=0.015, drift_eps=0.04

Mean-reversion / momentum diagnostics:
  H(mu)=0.978  | H(beta)=0.523 | H(residual)=0.193
  Residual ADF p=0 | KPSS p=0.1

Signal share:


C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\2546577285.py:89: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_resid_p = kpss(pd.Series(resid).dropna().values, regression="c", nlags="auto")[1]


BUY     0.418049
SELL    0.410270
FLAT    0.171680
Name: share, dtype: float64

,tick,price,fair_value,beta,residual,buy_signal,sell_signal,flat_signal,price_scaled,fv_scaled,residual_z
0,0,10010.0,10010.000000,0.000000,0.000000,False,False,True,100.1,100.100000,NaN
1,1,10000.0,10001.174787,-6.801602,-1.174787,False,True,False,100.0,100.011748,NaN
2,2,10000.0,9998.860996,-4.435393,1.139004,False,True,False,100.0,99.988610,NaN
3,3,10000.0,9998.396913,-2.897729,1.603087,False,True,False,100.0,99.983969,NaN
4,4,10000.0,9998.422530,-2.006580,1.577470,False,True,False,100.0,99.984225,NaN


Done: using fixed kalman_market_making params to visualize moving fair value + momentum.


In [ ]:
# Plot first 200 ticks: Price vs Kalman Fair Value
_slice = fair_df.head(200)

fig_fv = go.Figure()
fig_fv.add_trace(go.Scatter(
    x=_slice["tick"], y=_slice["price"],
    mode="lines", name="Price",
    line=dict(color="steelblue", width=1),
))
fig_fv.add_trace(go.Scatter(
    x=_slice["tick"], y=_slice["fair_value"],
    mode="lines", name="Kalman Fair Value (mu)",
    line=dict(color="orange", width=2.5),
))
fig_fv.update_layout(
    title="First 200 Ticks: Price vs Kalman Fair Value",
    xaxis_title="Tick",
    yaxis_title="Price",
    width=1100,
    height=500,
    hovermode="x unified",
    template="plotly_white",
)
fig_fv.show()

In [19]:
# =============================================================================
# Fixed mean (10_000) vs Kalman FV — is mean-reversion to a constant plausible?
# Run after cells 0–1 (full_prices). Uses same Kalman as kalman_fv.py.
# =============================================================================
import warnings
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.regression.linear_model import OLS
import statsmodels.api as sm
from plotly.subplots import make_subplots
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

FIXED_MEAN = 10_000.0
KF_R_OBS, KF_Q_LEVEL, KF_Q_DRIFT = 1.0, 0.3, 0.015


def _hurst_rs(ts, min_window=20):
    ts = np.asarray(ts, float)
    n = len(ts)
    max_k = int(np.floor(n / min_window))
    if max_k < 2:
        return np.nan
    ns, rs_vals = [], []
    for k in range(2, max_k + 1):
        size = n // k
        if size < min_window:
            break
        rs_list = []
        for i in range(k):
            chunk = ts[i * size : (i + 1) * size]
            devs = np.cumsum(chunk - chunk.mean())
            R = devs.max() - devs.min()
            S = chunk.std(ddof=1)
            if S > 0:
                rs_list.append(R / S)
        if rs_list:
            ns.append(size)
            rs_vals.append(np.mean(rs_list))
    if len(ns) < 3:
        return np.nan
    return np.polyfit(np.log(ns), np.log(rs_vals), 1)[0]


def _kalman_path(z, r_obs, q_level, q_drift):
    z = np.asarray(z, dtype=float)
    z = z[np.isfinite(z)]
    mu = np.zeros_like(z)
    beta = np.zeros_like(z)
    mu[0], beta[0] = z[0], 0.0
    p00, p01, p10, p11 = 25.0, 0.0, 0.0, 4.0
    F = np.array([[1.0, 1.0], [0.0, 1.0]])
    H = np.array([[1.0, 0.0]])
    Q = np.array([[q_level, 0.0], [0.0, q_drift]])
    for t in range(1, len(z)):
        x = np.array([mu[t - 1], beta[t - 1]])
        P = np.array([[p00, p01], [p10, p11]])
        x_pred = F @ x
        P_pred = F @ P @ F.T + Q
        innov = z[t] - np.squeeze(H @ x_pred)
        S = np.squeeze(H @ P_pred @ H.T) + r_obs
        K = (P_pred @ H.T).flatten() / S
        x_new = x_pred + K * innov
        P_new = (np.eye(2) - np.outer(K, H)) @ P_pred
        mu[t], beta[t] = x_new[0], x_new[1]
        p00, p01, p10, p11 = P_new[0, 0], P_new[0, 1], P_new[1, 0], P_new[1, 1]
    return mu, beta


def _ou_half_life(y):
    y = np.asarray(y, float)
    if len(y) < 30:
        return np.nan, np.nan
    dy = np.diff(y)
    ylag = y[:-1]
    res = OLS(dy, sm.add_constant(ylag)).fit()
    th = -res.params[1]
    if th <= 0:
        return np.inf, th
    return np.log(2) / th, th


def _safe_adf_kpss(x):
    x = pd.Series(x).replace([np.inf, -np.inf], np.nan).dropna().values
    if len(x) < 40:
        return np.nan, np.nan
    try:
        adf_p = adfuller(x, regression="c", autolag="AIC")[1]
    except Exception:
        adf_p = np.nan
    try:
        kpss_p = kpss(x, regression="c", nlags="auto")[1]
    except Exception:
        kpss_p = np.nan
    return adf_p, kpss_p


ash = full_prices[full_prices["product"] == "ASH_COATED_OSMIUM"].copy()
ash = ash.sort_values(["day", "timestamp"]).reset_index(drop=True)
mid = pd.to_numeric(ash["mid_price"], errors="coerce").replace(0.0, np.nan).dropna().values

dev_fixed = mid - FIXED_MEAN
mu_k, beta_k = _kalman_path(mid, KF_R_OBS, KF_Q_LEVEL, KF_Q_DRIFT)
resid_kalman = mid - mu_k

lr = np.diff(np.log(mid))

rows = []
for name, ser in [
    ("mid level", mid),
    ("fixed anchor (mid-10k)", dev_fixed),
    ("Kalman residual (mid-mu)", resid_kalman),
    ("log return", lr),
]:
    s = pd.Series(ser).replace([np.inf, -np.inf], np.nan).dropna().values
    adf_p, kpss_p = _safe_adf_kpss(s)
    hl, th = _ou_half_life(s) if name != "log return" else (np.nan, np.nan)
    rows.append({
        "series": name,
        "n": len(s),
        "hurst": round(_hurst_rs(s), 4) if len(s) > 200 else np.nan,
        "adf_p": round(adf_p, 4) if pd.notna(adf_p) else np.nan,
        "kpss_p": round(kpss_p, 4) if pd.notna(kpss_p) else np.nan,
        "ou_half_life_ticks": round(hl, 2) if np.isfinite(hl) else None,
        "ou_theta": round(th, 6) if pd.notna(th) and np.isfinite(th) else None,
    })

compare_df = pd.DataFrame(rows)
print("Interpretation: for MR-to-10k, want (mid-10k) stationary / low Hurst / finite OU HL.")
print("Kalman residual tests whether MR is vs *moving* FV instead.\n")
display(compare_df)

fig = go.Figure()
fig.add_trace(go.Scatter(y=mid, mode="lines", name="mid", line=dict(width=1, color="steelblue")))
fig.add_trace(go.Scatter(y=mu_k, mode="lines", name="Kalman mu", line=dict(width=1.5, color="orange")))
fig.add_hline(y=FIXED_MEAN, line_dash="dash", line_color="green", annotation_text="fixed 10k")
fig.update_layout(title="ASH_COATED_OSMIUM: mid vs Kalman FV vs fixed 10k", height=450, template="plotly_white")
fig.show()

fig2 = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45], vertical_spacing=0.08,
    subplot_titles=("Deviation from 10k", "Kalman residual (mid - mu)"))
fig2.add_trace(go.Scatter(y=dev_fixed, mode="lines", name="mid - 10k", line=dict(color="purple", width=1)), row=1, col=1)
fig2.add_hline(y=0, line_dash="dot", line_color="black", row=1, col=1)
fig2.add_trace(go.Scatter(y=resid_kalman, mode="lines", name="mid - mu_k", line=dict(color="teal", width=1)), row=2, col=1)
fig2.add_hline(y=0, line_dash="dot", line_color="black", row=2, col=1)
fig2.update_layout(height=600, template="plotly_white", title_text="Anchors for mean-reversion")
fig2.show()

C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\712135914.py:92: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = kpss(x, regression="c", nlags="auto")[1]
C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\712135914.py:92: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_p = kpss(x, regression="c", nlags="auto")[1]
C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\712135914.py:92: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_p = kpss(x, regression="c", nlags="auto")[1]
C:\Users\alexn\AppData\Local\Temp\ipykernel_27476\712135914.py:92: InterpolationWarning: The test statistic is outside of the range of p-value

Interpretation: for MR-to-10k, want (mid-10k) stationary / low Hurst / finite OU HL.
Kalman residual tests whether MR is vs *moving* FV instead.



,series,n,hurst,adf_p,kpss_p,ou_half_life_ticks,ou_theta
0,mid level,29951,1.0168,0.0,0.01,2.86,0.242744
1,fixed anchor (mid-10k),29951,1.0168,0.0,0.01,2.86,0.242744
2,Kalman residual (mid-mu),29951,0.1867,0.0,0.10,0.57,1.217129
3,log return,29950,0.2100,0.0,0.10,NaN,NaN


Interpretation: for MR-to-10k, want (mid-10k) stationary / low Hurst / finite OU HL.
Kalman residual tests whether MR is vs *moving* FV instead.



,series,n,hurst,adf_p,kpss_p,ou_half_life_ticks,ou_theta
0,mid level,29951,1.0168,0.0,0.01,2.86,0.242744
1,fixed anchor (mid-10k),29951,1.0168,0.0,0.01,2.86,0.242744
2,Kalman residual (mid-mu),29951,0.1867,0.0,0.10,0.57,1.217129
3,log return,29950,0.2100,0.0,0.10,NaN,NaN


In [21]:
# =============================================================================
# 2-state Markov switching (statsmodels) — regime in variance / mean of returns
# Prefer log returns (numerically stable). Raw (mid-10k) often blows up — avoid.
# =============================================================================
from plotly.subplots import make_subplots
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

MAX_RET_POINTS = 8000  # cap for speed; set None for full series
MARKOV_K = 3         # number of Markov states/regimes

ret = pd.Series(np.diff(np.log(mid))).replace([np.inf, -np.inf], np.nan).dropna()
if MAX_RET_POINTS is not None and len(ret) > MAX_RET_POINTS:
    ret_fit = ret.iloc[-MAX_RET_POINTS:]
    print(f"Fitting MarkovRegression on last {len(ret_fit)} log returns (of {len(ret)})")
else:
    ret_fit = ret

mod = MarkovRegression(ret_fit.values, k_regimes=MARKOV_K, trend="c", switching_variance=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res = mod.fit(search_reps=20)

print(res.summary())

smoothed = res.smoothed_marginal_probabilities
# statsmodels version differences: can be DataFrame-like or ndarray
if hasattr(smoothed, "values"):
    sm_arr = np.asarray(smoothed.values)
else:
    sm_arr = np.asarray(smoothed)

# For k>2, use the last regime probability for visualization.
if sm_arr.ndim == 2 and sm_arr.shape[1] > 1:
    p_reg1 = sm_arr[:, -1]
else:
    p_reg1 = sm_arr.reshape(-1)

# Print a clean transition matrix (for k=2, statsmodels often only prints p[0->0], p[1->0])
try:
    p00 = float(res.params["p[0->0]"])
    p10 = float(res.params["p[1->0]"])
except Exception:
    # Fallback: parse from params vector by name list if available
    p00 = p10 = None

if p00 is not None and p10 is not None:
    P = np.array([[p00, 1.0 - p00], [p10, 1.0 - p10]])
    print("\nTransition matrix P (rows=from, cols=to):")
    print(P)
    # expected duration in a regime ~ 1 / (1 - p_ii)
    d0 = 1.0 / (1.0 - P[0, 0]) if P[0, 0] < 1 else np.inf
    d1 = 1.0 / (1.0 - P[1, 1]) if P[1, 1] < 1 else np.inf
    print(f"Expected duration: regime0 ~ {d0:.2f} ticks, regime1 ~ {d1:.2f} ticks")

p_high_var = p_reg1

t_ret = np.arange(len(ret_fit))

figm = make_subplots(rows=3, cols=1, shared_xaxes=True, row_heights=[0.45, 0.25, 0.3], vertical_spacing=0.06,
    subplot_titles=("Log return", "Smoothed P(regime 1)", "Mid price (aligned to return window)"))
figm.add_trace(go.Scatter(x=t_ret, y=ret_fit.values, mode="lines", name="d log mid", line=dict(width=1)), row=1, col=1)
figm.add_trace(go.Scatter(x=t_ret, y=p_high_var, mode="lines", name="P(reg1)", line=dict(width=1.5, color="crimson")), row=2, col=1)
# mid aligned: same length as returns window (last len(ret_fit)+1 mids)
mid_win = mid[-(len(ret_fit) + 1) :]
figm.add_trace(go.Scatter(x=np.arange(len(mid_win)), y=mid_win, mode="lines", name="mid", line=dict(width=1)), row=3, col=1)
figm.update_layout(height=780, template="plotly_white", title_text="Markov switching on log returns")
figm.show()

# Optional: standardized deviation from 10k (often more stable than raw scale)
z = (dev_fixed - np.nanmean(dev_fixed)) / (np.nanstd(dev_fixed) + 1e-12)
z = pd.Series(z).replace([np.inf, -np.inf], np.nan).dropna()
MAX_Z = 5000
z_fit = z.iloc[-MAX_Z:] if len(z) > MAX_Z else z
print(f"\n--- Alt spec: Markov on standardized (mid-10k), n={len(z_fit)} ---")
mod2 = MarkovRegression(z_fit.values, k_regimes=MARKOV_K, trend="c", switching_variance=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res2 = mod2.fit(search_reps=15)
print(res2.summary())
sm2 = res2.smoothed_marginal_probabilities
if hasattr(sm2, "values"):
    sm2_arr = np.asarray(sm2.values)
else:
    sm2_arr = np.asarray(sm2)

if sm2_arr.ndim == 2 and sm2_arr.shape[1] > 1:
    p2 = sm2_arr[:, 1]
else:
    p2 = sm2_arr.reshape(-1)
tz = np.arange(len(z_fit))
figz = make_subplots(rows=2, cols=1, shared_xaxes=True, row_heights=[0.55, 0.45], vertical_spacing=0.08)
figz.add_trace(go.Scatter(x=tz, y=z_fit.values, mode="lines", name="z(mid-10k)", line=dict(width=1)), row=1, col=1)
figz.add_trace(go.Scatter(x=tz, y=p2, mode="lines", name="P(reg1)", line=dict(width=1.5, color="darkorange")), row=2, col=1)
figz.update_layout(height=520, template="plotly_white", title_text="Markov on standardized deviation from 10k")
figz.show()

Fitting MarkovRegression on last 8000 log returns (of 29950)
                        Markov Switching Model Results                        
Dep. Variable:                      y   No. Observations:                 8000
Model:               MarkovRegression   Log Likelihood               54982.296
Date:                Wed, 15 Apr 2026   AIC                        -109952.591
Time:                        12:28:40   BIC                        -109910.668
Sample:                             0   HQIC                       -109938.241
                               - 8000                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const      -9.073e-07   1.54e-06     -0.590      0.555   -3.92e-06    


--- Alt spec: Markov on standardized (mid-10k), n=5000 ---
                        Markov Switching Model Results                        
Dep. Variable:                      y   No. Observations:                 5000
Model:               MarkovRegression   Log Likelihood               -6183.427
Date:                Wed, 15 Apr 2026   AIC                          12378.853
Time:                        12:28:41   BIC                          12417.956
Sample:                             0   HQIC                         12392.558
                               - 5000                                         
Covariance Type:               approx                                         
                             Regime 0 parameters                              
                 coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------
const         -0.7962      0.028    -28.319      0.000      -0.851     

In [23]:
# =============================================================================
# Volatility regime visualisation — evidence for the regime switch
# Re-uses the Markov fit from cell 9 (res, ret_fit, mid).
# =============================================================================
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# --- Smoothed regime probability from the first Markov fit (log returns) ---
sm = res.smoothed_marginal_probabilities
if hasattr(sm, "values"):
    sm_arr = sm.values
else:
    sm_arr = np.asarray(sm)

p_storm = sm_arr[:, 1] if sm_arr.ndim == 2 and sm_arr.shape[1] > 1 else sm_arr.reshape(-1)

# Align: returns are 1 shorter than the mid window
mid_window = mid[-(len(ret_fit) + 1):]
t = np.arange(len(ret_fit))

# --- Squared returns (raw instantaneous variance proxy) ---
ret_sq = ret_fit.values ** 2

fig = make_subplots(
    rows=4, cols=1, shared_xaxes=True,
    row_heights=[0.30, 0.20, 0.20, 0.30],
    vertical_spacing=0.04,
    subplot_titles=(
        "ASH Mid Price (coloured by regime)",
        "Log Return",
        "Squared Return (instantaneous variance proxy)",
        "P(high-vol regime) — smoothed Markov probability",
    ),
)

# Row 1: mid price, coloured by regime probability
# Split into quiet (blue) and storm (red) segments via marker colour
colors = [f"rgba(220,50,50,{max(p, 0.15):.2f})" if p > 0.5
          else f"rgba(70,130,180,{max(1-p, 0.15):.2f})"
          for p in p_storm]
fig.add_trace(go.Scattergl(
    x=t, y=mid_window[1:], mode="markers+lines",
    marker=dict(color=colors, size=2),
    line=dict(color="lightgray", width=0.5),
    name="mid (coloured by regime)",
    showlegend=False,
), row=1, col=1)

# Row 2: log returns
fig.add_trace(go.Scattergl(
    x=t, y=ret_fit.values, mode="lines",
    line=dict(color="gray", width=0.8), name="log return", showlegend=False,
), row=2, col=1)
fig.add_hline(y=0, line_dash="dot", line_color="black", row=2, col=1)

# Row 3: squared returns — spikes = high-vol regime
fig.add_trace(go.Scattergl(
    x=t, y=ret_sq, mode="lines",
    line=dict(color="darkorange", width=1), name="ret²", showlegend=False,
), row=3, col=1)
# Overlay the two regime variance levels as horizontal lines
_pv = res.params.values if hasattr(res.params, "values") else res.params
sig2_0, sig2_1 = float(_pv[1]), float(_pv[3])
fig.add_hline(y=sig2_0, line_dash="dash", line_color="steelblue",
              annotation_text=f"quiet ({sig2_0:.2e})", row=3, col=1)
fig.add_hline(y=sig2_1, line_dash="dash", line_color="crimson",
              annotation_text=f"storm ({sig2_1:.2e})", row=3, col=1)

# Row 4: smoothed regime probability
fig.add_trace(go.Scattergl(
    x=t, y=p_storm, mode="lines",
    line=dict(color="crimson", width=1.5), name="P(storm)", showlegend=False,
), row=4, col=1)
fig.add_hline(y=0.5, line_dash="dot", line_color="black", row=4, col=1)

fig.update_yaxes(title_text="Mid Price", row=1, col=1)
fig.update_yaxes(title_text="Log Ret", row=2, col=1)
fig.update_yaxes(title_text="Ret²", row=3, col=1)
fig.update_yaxes(title_text="P(high-vol)", row=4, col=1)
fig.update_xaxes(title_text="Tick (within window)", row=4, col=1)

fig.update_layout(
    height=950, width=1200,
    title="Volatility Regime Switch — ASH_COATED_OSMIUM",
    template="plotly_white",
    hovermode="x unified",
)
fig.show()

# --- Summary stats ---
n_storm = (p_storm > 0.5).sum()
n_quiet = (p_storm <= 0.5).sum()
pct_storm = 100 * n_storm / len(p_storm)
print(f"Ticks classified as high-vol (P>0.5): {n_storm}/{len(p_storm)} ({pct_storm:.1f}%)")
print(f"Regime 0 sigma2 = {sig2_0:.2e}  (quiet)")
print(f"Regime 1 sigma2 = {sig2_1:.2e}  (storm)")
print(f"Ratio: {sig2_1 / sig2_0:.1f}x")

In [24]:
# =============================================================================
# Micro-regime analysis — zoom into regime episodes
# Uses smoothed Markov probs from cell 9 (res, ret_fit, mid, p_storm from cell 10)
# =============================================================================
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── Identify regime episodes (contiguous runs of quiet / storm) ───────────
threshold = 0.5
regime_label = (p_storm > threshold).astype(int)  # 0=quiet, 1=storm

# Find episode boundaries
episodes = []
start = 0
for i in range(1, len(regime_label)):
    if regime_label[i] != regime_label[start]:
        episodes.append({
            "start": start, "end": i - 1,
            "regime": int(regime_label[start]),
            "duration": i - start,
            "mean_p": float(p_storm[start:i].mean()),
        })
        start = i
episodes.append({
    "start": start, "end": len(regime_label) - 1,
    "regime": int(regime_label[start]),
    "duration": len(regime_label) - start,
    "mean_p": float(p_storm[start:].mean()),
})

ep_df = pd.DataFrame(episodes)
storm_eps = ep_df[ep_df["regime"] == 1]
quiet_eps = ep_df[ep_df["regime"] == 0]

print(f"Total episodes: {len(ep_df)}  ({len(quiet_eps)} quiet, {len(storm_eps)} storm)")
print(f"\nStorm episode durations:")
print(storm_eps["duration"].describe().to_string())
print(f"\nQuiet episode durations:")
print(quiet_eps["duration"].describe().to_string())

# ── Fig 1: Duration histogram — how long do micro-regimes last? ───────────
fig1 = make_subplots(rows=1, cols=2, subplot_titles=("Quiet episode durations", "Storm episode durations"))
fig1.add_trace(go.Histogram(
    x=quiet_eps["duration"], nbinsx=50,
    marker_color="steelblue", name="Quiet",
), row=1, col=1)
fig1.add_trace(go.Histogram(
    x=storm_eps["duration"], nbinsx=50,
    marker_color="crimson", name="Storm",
), row=1, col=2)
fig1.update_xaxes(title_text="Duration (ticks)", row=1, col=1)
fig1.update_xaxes(title_text="Duration (ticks)", row=1, col=2)
fig1.update_yaxes(title_text="Count", row=1, col=1)
fig1.update_layout(height=400, width=1100, title="Micro-regime durations", template="plotly_white", showlegend=False)
fig1.show()

# ── Fig 2: Price with regime bands (shaded storm episodes) ────────────────
mid_w = mid[-(len(ret_fit) + 1):]  # aligned mid window
t_mid = np.arange(len(mid_w))

fig2 = go.Figure()
fig2.add_trace(go.Scattergl(
    x=t_mid, y=mid_w, mode="lines",
    line=dict(color="steelblue", width=1), name="Mid price",
))
# Shade storm episodes
for _, ep in storm_eps.iterrows():
    s, e = int(ep["start"]) + 1, int(ep["end"]) + 1  # +1 offset for mid alignment
    fig2.add_vrect(x0=s, x1=e, fillcolor="red", opacity=0.15, line_width=0)

fig2.update_layout(
    height=450, width=1200,
    title="ASH mid price — red bands = high-vol regime episodes",
    xaxis_title="Tick", yaxis_title="Mid Price",
    template="plotly_white",
)
fig2.show()

# ── Fig 3: Zoom into a few storm episodes to see micro-structure ──────────
# Pick the 5 longest storm episodes for close-up
top_storms = storm_eps.nlargest(min(6, len(storm_eps)), "duration")

n_panels = len(top_storms)
fig3 = make_subplots(
    rows=n_panels, cols=1, shared_xaxes=False,
    vertical_spacing=0.08,
    subplot_titles=[f"Storm ep @ tick {int(r['start'])} (dur={int(r['duration'])})" for _, r in top_storms.iterrows()],
)

for i, (_, ep) in enumerate(top_storms.iterrows()):
    pad = max(int(ep["duration"]), 10)  # context before/after
    s = max(int(ep["start"]) + 1 - pad, 0)
    e = min(int(ep["end"]) + 1 + pad, len(mid_w) - 1)
    t_slice = np.arange(s, e + 1)

    # Price
    fig3.add_trace(go.Scatter(
        x=t_slice, y=mid_w[s:e + 1], mode="lines",
        line=dict(color="steelblue", width=1.5), showlegend=False,
    ), row=i + 1, col=1)

    # Shade the storm portion
    fig3.add_vrect(
        x0=int(ep["start"]) + 1, x1=int(ep["end"]) + 1,
        fillcolor="red", opacity=0.2, line_width=0,
        row=i + 1, col=1,
    )

fig3.update_layout(
    height=250 * n_panels, width=1100,
    title="Zoomed into longest storm micro-regimes (red = high-vol)",
    template="plotly_white",
)
fig3.show()

# ── Fig 4: Regime-conditional return distributions ────────────────────────
ret_vals = ret_fit.values
quiet_rets = ret_vals[regime_label == 0]
storm_rets = ret_vals[regime_label == 1]

fig4 = go.Figure()
fig4.add_trace(go.Histogram(
    x=quiet_rets, nbinsx=200, histnorm="probability density",
    marker_color="steelblue", opacity=0.6, name=f"Quiet (n={len(quiet_rets)})",
))
fig4.add_trace(go.Histogram(
    x=storm_rets, nbinsx=200, histnorm="probability density",
    marker_color="crimson", opacity=0.6, name=f"Storm (n={len(storm_rets)})",
))
fig4.update_layout(
    barmode="overlay",
    height=450, width=1000,
    title="Return distribution by regime — storm has much fatter tails",
    xaxis_title="Log return", yaxis_title="Density",
    template="plotly_white",
)
fig4.show()

print(f"\nQuiet returns: mean={quiet_rets.mean():.2e}, std={quiet_rets.std():.2e}")
print(f"Storm returns: mean={storm_rets.mean():.2e}, std={storm_rets.std():.2e}")
print(f"Std ratio (storm/quiet): {storm_rets.std() / quiet_rets.std():.1f}×")

Total episodes: 1552  (776 quiet, 776 storm)

Storm episode durations:
count    776.000000
mean       3.202320
std        2.180629
min        1.000000
25%        2.000000
50%        2.000000
75%        4.000000
max       17.000000

Quiet episode durations:
count    776.000000
mean       7.106959
std        5.967236
min        1.000000
25%        3.000000
50%        5.000000
75%        9.000000
max       36.000000



Quiet returns: mean=-4.80e-07, std=9.46e-05
Storm returns: mean=1.15e-06, std=6.40e-04
Std ratio (storm/quiet): 6.8×


In [ ]:
# =============================================================================
# What triggers storms? Distance from mean, price level, or something else?
# Uses p_storm, mid, mu_k (Kalman FV), regime_label from earlier cells.
# =============================================================================
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Align everything to the return window (last 8001 mids → 8000 returns)
mid_w = mid[-(len(ret_fit) + 1):]
mu_w = mu_k[-(len(ret_fit) + 1):]  # Kalman FV aligned
ret_vals = ret_fit.values

# Distance from fixed 10k
dist_10k = mid_w[1:] - 10_000.0

# Distance from Kalman FV (residual)
dist_kalman = mid_w[1:] - mu_w[1:]

# Absolute distance (magnitude)
abs_dist_10k = np.abs(dist_10k)
abs_dist_kalman = np.abs(dist_kalman)

# Price level itself
price_level = mid_w[1:]

# ── Fig 1: Scatter — does storm happen when far from 10k? ────────────────
fig1 = make_subplots(rows=1, cols=3, subplot_titles=(
    "Storm prob vs distance from 10k",
    "Storm prob vs distance from Kalman FV",
    "Storm prob vs price level",
))

fig1.add_trace(go.Scattergl(
    x=dist_10k, y=p_storm, mode="markers",
    marker=dict(size=2, color=regime_label, colorscale=["steelblue", "crimson"], opacity=0.4),
    showlegend=False,
), row=1, col=1)

fig1.add_trace(go.Scattergl(
    x=dist_kalman, y=p_storm, mode="markers",
    marker=dict(size=2, color=regime_label, colorscale=["steelblue", "crimson"], opacity=0.4),
    showlegend=False,
), row=1, col=2)

fig1.add_trace(go.Scattergl(
    x=price_level, y=p_storm, mode="markers",
    marker=dict(size=2, color=regime_label, colorscale=["steelblue", "crimson"], opacity=0.4),
    showlegend=False,
), row=1, col=3)

fig1.update_xaxes(title_text="mid − 10k", row=1, col=1)
fig1.update_xaxes(title_text="mid − Kalman μ", row=1, col=2)
fig1.update_xaxes(title_text="mid price", row=1, col=3)
fig1.update_yaxes(title_text="P(storm)", row=1, col=1)
fig1.update_layout(height=400, width=1300, title="Does storm correlate with distance from mean?", template="plotly_white")
fig1.show()

# ── Fig 2: Binned analysis — avg P(storm) by |distance| bucket ───────────
def _binned_prob(x, p, n_bins=20):
    bins = np.linspace(np.nanpercentile(x, 1), np.nanpercentile(x, 99), n_bins + 1)
    centers, avg_p, counts = [], [], []
    for i in range(len(bins) - 1):
        mask = (x >= bins[i]) & (x < bins[i + 1])
        if mask.sum() > 10:
            centers.append((bins[i] + bins[i + 1]) / 2)
            avg_p.append(float(p[mask].mean()))
            counts.append(int(mask.sum()))
    return np.array(centers), np.array(avg_p), np.array(counts)

c1, p1, n1 = _binned_prob(abs_dist_10k, p_storm)
c2, p2, n2 = _binned_prob(abs_dist_kalman, p_storm)
c3, p3, n3 = _binned_prob(np.abs(ret_vals), p_storm)

fig2 = make_subplots(rows=1, cols=3, subplot_titles=(
    "Avg P(storm) by |mid − 10k|",
    "Avg P(storm) by |mid − Kalman μ|",
    "Avg P(storm) by |log return|",
))

fig2.add_trace(go.Bar(x=c1, y=p1, marker_color="mediumpurple", showlegend=False), row=1, col=1)
fig2.add_trace(go.Bar(x=c2, y=p2, marker_color="teal", showlegend=False), row=1, col=2)
fig2.add_trace(go.Bar(x=c3, y=p3, marker_color="darkorange", showlegend=False), row=1, col=3)

fig2.update_yaxes(title_text="Avg P(storm)", row=1, col=1)
fig2.update_xaxes(title_text="|mid − 10k|", row=1, col=1)
fig2.update_xaxes(title_text="|mid − Kalman μ|", row=1, col=2)
fig2.update_xaxes(title_text="|log return|", row=1, col=3)
fig2.update_layout(height=400, width=1300, title="Binned: where do storms happen?", template="plotly_white")
fig2.show()

# ── Fig 3: Time-lagged — do storms FOLLOW big moves or PRECEDE them? ─────
max_lag = 10
lag_corrs = []
for lag in range(-max_lag, max_lag + 1):
    if lag >= 0:
        x = abs_dist_kalman[lag:]
        y = p_storm[:len(x)]
    else:
        x = abs_dist_kalman[:lag]
        y = p_storm[-lag:]
    if len(x) > 100:
        corr = float(np.corrcoef(x, y)[0, 1])
    else:
        corr = 0.0
    lag_corrs.append({"lag": lag, "corr": corr})

lag_df = pd.DataFrame(lag_corrs)

fig3 = go.Figure()
fig3.add_trace(go.Bar(
    x=lag_df["lag"], y=lag_df["corr"],
    marker_color=["crimson" if c > 0 else "steelblue" for c in lag_df["corr"]],
))
fig3.add_vline(x=0, line_dash="dot", line_color="black")
fig3.update_layout(
    height=400, width=900,
    title="Cross-correlation: |Kalman residual| vs P(storm) at different lags<br>"
          "<sub>Negative lag = storm FOLLOWS displacement; Positive = storm PRECEDES displacement</sub>",
    xaxis_title="Lag (ticks): negative = residual leads storm",
    yaxis_title="Correlation",
    template="plotly_white",
)
fig3.show()

# ── Summary ───────────────────────────────────────────────────────────────
corr_10k = float(np.corrcoef(abs_dist_10k, p_storm)[0, 1])
corr_kalman = float(np.corrcoef(abs_dist_kalman, p_storm)[0, 1])
corr_ret = float(np.corrcoef(np.abs(ret_vals), p_storm)[0, 1])
print(f"Correlation of P(storm) with:")
print(f"  |mid − 10k|      : {corr_10k:+.4f}")
print(f"  |mid − Kalman μ| : {corr_kalman:+.4f}")
print(f"  |log return|     : {corr_ret:+.4f}")
print()
if corr_ret > max(corr_10k, corr_kalman) + 0.05:
    print("→ Storms are driven by LARGE MOVES (return size), not by distance from any mean.")
elif corr_10k > corr_ret + 0.05:
    print("→ Storms tend to happen when price is FAR FROM 10k.")
elif corr_kalman > corr_ret + 0.05:
    print("→ Storms tend to happen when price is FAR FROM Kalman FV.")
else:
    print("→ Mixed signal — storms are somewhat related to both move size and displacement.")

In [ ]:
# =============================================================================
# Inventory-curve simulation for pure_as_regime.py strategy
# Uses historical best-bid / best-ask for fill simulation
# =============================================================================
import math
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# ── Strategy params (mirror pure_as_regime.py exactly) ────────────────────
POSITION_LIMIT = 80;  PEPPER_LIMIT = 80;  T_MAX = 999_900

# ASH
VAR_WIN = 20;   MIN_VAR_OBS = 5
BL_ALPHA = 0.025;  SP_WIN = 3;  SP_TH = 4.0;  SP_CEIL = 10.0
GAMMA_Q = 0.05;  GAMMA_S = 0.20;  K_ARR = 0.40;  TAU_MIN = 0.005
MIN_SP = 2;  STORM_LIM = 35;  INV_ASYM = 0.30
TAKE_EDGE = 2.0;  TAKE_MAX_FRAC = 0.75;  TAKE_REGIME_GATE = 0.3
EOD_TAU = 0.10;  EOD_INV_PEN = 4.0

# PEPPER
PP_GAMMA = 0.03;  PP_K = 0.40;  PP_MIN_SP = 2;  PP_VAR_WIN = 20
PP_LONG_TGT = 0.75;  PP_BID_TIGHT = 1.5;  PP_ASK_WIDE = 1.5

def _ewma(prev, val, alpha):
    return val if prev is None else alpha * val + (1 - alpha) * prev

def _spike_regime(worst, base, th, ceil):
    if base < 1e-15: return 0.0
    r = worst / base
    if r <= th: return 0.0
    if r >= ceil: return 1.0
    return (r - th) / (ceil - th)

def _safe_mid(best_bid, best_ask, bid2, ask2):
    if not np.isnan(bid2) and not np.isnan(ask2):
        return (bid2 + ask2) / 2.0
    return (best_bid + best_ask) / 2.0


# ── Simulate ASH ─────────────────────────────────────────────────────────
def simulate_ash(df):
    pos = 0;  cash = 0.0
    mid_buf = [];  recent_dp2 = [];  baseline_var = 1.0;  last_mid = None
    inv, pnl, phv_list, spread_list = [], [], [], []

    for _, row in df.iterrows():
        best_bid = row["bid_price_1"];  best_ask = row["ask_price_1"]
        bid2 = row.get("bid_price_2", np.nan);  ask2 = row.get("ask_price_2", np.nan)
        ts = row["timestamp"]
        if np.isnan(best_bid) or np.isnan(best_ask): continue
        mid = _safe_mid(best_bid, best_ask, bid2, ask2)

        mid_buf.append(mid)
        mid_buf = mid_buf[-(VAR_WIN + 1):]
        rolling_var = max(float(np.var(np.diff(mid_buf))), 1e-6) if len(mid_buf) >= MIN_VAR_OBS + 1 else 1.0

        if last_mid is not None:
            dp2 = (mid - last_mid) ** 2
            baseline_var = _ewma(baseline_var, dp2, BL_ALPHA)
            recent_dp2.append(dp2)
            recent_dp2 = recent_dp2[-SP_WIN:]
        last_mid = mid

        worst = max(recent_dp2) if recent_dp2 else baseline_var
        p_hv = _spike_regime(worst, baseline_var, SP_TH, SP_CEIL)

        gamma = GAMMA_Q + (GAMMA_S - GAMMA_Q) * p_hv
        plim = POSITION_LIMIT - (POSITION_LIMIT - STORM_LIM) * p_hv
        sig2 = max((1 - p_hv) * rolling_var + p_hv * worst, 1e-6)
        tau = max(1 - ts / T_MAX, TAU_MIN)

        reservation = mid - pos * gamma * sig2 * tau
        if tau < EOD_TAU:
            urgency = (EOD_TAU - tau) / EOD_TAU
            reservation -= EOD_INV_PEN * urgency * pos / POSITION_LIMIT

        spread = gamma * sig2 * tau + (2 / gamma) * math.log(1 + gamma / K_ARR)
        spread = max(spread, MIN_SP)
        qn = pos / max(plim, 1)
        hb = (spread / 2) * (1 + INV_ASYM * max(0, qn))
        ha = (spread / 2) * (1 + INV_ASYM * max(0, -qn))
        as_bid = math.floor(reservation - hb)
        as_ask = math.ceil(reservation + ha)

        buy_cap = int(plim) - pos
        sell_cap = -int(plim) - pos

        # Tactical taking
        can_take = (p_hv < TAKE_REGIME_GATE and abs(pos) < TAKE_MAX_FRAC * plim)
        if can_take:
            if best_ask < mid - TAKE_EDGE and buy_cap > 0:
                qty = min(abs(int(row.get("ask_volume_1", 0) or 0)), buy_cap)
                if qty > 0:
                    pos += qty; cash -= best_ask * qty; buy_cap -= qty
            if best_bid > mid + TAKE_EDGE and sell_cap < 0:
                qty = min(abs(int(row.get("bid_volume_1", 0) or 0)), -sell_cap)
                if qty > 0:
                    pos -= qty; cash += best_bid * qty; sell_cap += qty

        # Passive fills: if our bid >= best_ask -> we get filled buying
        mm_bid = min(as_bid, int(best_bid) + 1)
        mm_ask = max(as_ask, int(best_ask) - 1)
        if mm_bid >= best_ask and buy_cap > 0:
            fill = min(abs(int(row.get("ask_volume_1", 0) or 0)), buy_cap, 5)
            if fill > 0:
                pos += fill; cash -= best_ask * fill
        elif mm_ask <= best_bid and sell_cap < 0:
            fill = min(abs(int(row.get("bid_volume_1", 0) or 0)), -sell_cap, 5)
            if fill > 0:
                pos -= fill; cash += best_bid * fill

        inv.append(pos)
        pnl.append(cash + pos * mid)
        phv_list.append(p_hv)
        spread_list.append(spread)

    return inv, pnl, phv_list, spread_list


# ── Simulate PEPPER ──────────────────────────────────────────────────────
def simulate_pepper(df):
    pos = 0;  cash = 0.0;  mid_buf = []
    target = PP_LONG_TGT * PEPPER_LIMIT
    inv, pnl = [], []

    for _, row in df.iterrows():
        best_bid = row["bid_price_1"];  best_ask = row["ask_price_1"]
        ts = row["timestamp"]
        if np.isnan(best_bid) or np.isnan(best_ask): continue
        bid2 = row.get("bid_price_2", np.nan);  ask2 = row.get("ask_price_2", np.nan)
        mid = _safe_mid(best_bid, best_ask, bid2, ask2)

        mid_buf.append(mid)
        mid_buf = mid_buf[-(PP_VAR_WIN + 1):]
        sig2 = max(float(np.var(np.diff(mid_buf))), 1e-6) if len(mid_buf) >= 6 else 1.0
        tau = max(1 - ts / T_MAX, TAU_MIN)

        inv_gap = pos - target
        reservation = mid - inv_gap * PP_GAMMA * sig2 * tau
        nudge = max(-4.0, min(4.0, 0.02 * inv_gap))
        reservation -= nudge

        spread = PP_GAMMA * sig2 * tau + (2 / PP_GAMMA) * math.log(1 + PP_GAMMA / PP_K)
        spread = max(spread, PP_MIN_SP)
        hb = max(spread / 2 - PP_BID_TIGHT, 0.5)
        ha = spread / 2 + PP_ASK_WIDE

        pp_bid = math.floor(reservation - hb)
        pp_ask = math.ceil(reservation + ha)

        buy_cap = PEPPER_LIMIT - pos
        sell_cap = -PEPPER_LIMIT - pos

        mm_bid = min(pp_bid, int(best_bid) + 1)
        mm_ask = max(pp_ask, int(best_ask) - 1)

        if mm_bid >= best_ask and buy_cap > 0:
            fill = min(abs(int(row.get("ask_volume_1", 0) or 0)), buy_cap, 5)
            if fill > 0:
                pos += fill; cash -= best_ask * fill
        elif mm_ask <= best_bid and sell_cap < 0:
            fill = min(abs(int(row.get("bid_volume_1", 0) or 0)), -sell_cap, 5)
            if fill > 0:
                pos -= fill; cash += best_bid * fill

        inv.append(pos)
        pnl.append(cash + pos * mid)

    return inv, pnl


# ── Run both ─────────────────────────────────────────────────────────────
ash_sim = full_prices[full_prices["product"] == "ASH_COATED_OSMIUM"].sort_values(["day", "timestamp"]).reset_index(drop=True)
pp_sim  = full_prices[full_prices["product"] == "INTARIAN_PEPPER_ROOT"].sort_values(["day", "timestamp"]).reset_index(drop=True)

ash_inv, ash_pnl, ash_phv, ash_sprd = simulate_ash(ash_sim)
pp_inv, pp_pnl = simulate_pepper(pp_sim)

ash_mid_plot = pd.to_numeric(ash_sim["mid_price"], errors="coerce").replace(0, np.nan).dropna().values[:len(ash_inv)]
pp_mid_plot  = pd.to_numeric(pp_sim["mid_price"],  errors="coerce").replace(0, np.nan).dropna().values[:len(pp_inv)]

# ── Plot ASH ─────────────────────────────────────────────────────────────
t_a = np.arange(len(ash_inv))
fig_ash = make_subplots(
    rows=4, cols=1, shared_xaxes=True, vertical_spacing=0.05,
    row_heights=[0.30, 0.25, 0.20, 0.25],
    subplot_titles=("ASH Mid Price", "Inventory", "P(storm)", "Mark-to-Market PnL"),
)
fig_ash.add_trace(go.Scattergl(x=t_a, y=ash_mid_plot, mode="lines", line=dict(width=1, color="steelblue"), showlegend=False), row=1, col=1)
fig_ash.add_trace(go.Scattergl(x=t_a, y=ash_inv, mode="lines", line=dict(width=1.5, color="darkorange"), showlegend=False), row=2, col=1)
fig_ash.add_hline(y=0, line_dash="dot", line_color="grey", row=2, col=1)
fig_ash.add_hline(y=POSITION_LIMIT, line_dash="dash", line_color="red", annotation_text="+80 lim", row=2, col=1)
fig_ash.add_hline(y=-POSITION_LIMIT, line_dash="dash", line_color="red", annotation_text="-80 lim", row=2, col=1)
fig_ash.add_trace(go.Scattergl(x=t_a, y=ash_phv, mode="lines", line=dict(width=1, color="crimson"), showlegend=False), row=3, col=1)
fig_ash.add_trace(go.Scattergl(x=t_a, y=ash_pnl, mode="lines", line=dict(width=1.5, color="green"), showlegend=False), row=4, col=1)
fig_ash.add_hline(y=0, line_dash="dot", line_color="grey", row=4, col=1)
fig_ash.update_layout(height=900, width=1200, title="ASH - pure_as_regime Inventory & PnL Simulation", template="plotly_white")
fig_ash.show()

# ── Plot PEPPER ──────────────────────────────────────────────────────────
t_p = np.arange(len(pp_inv))
fig_pp = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    row_heights=[0.35, 0.30, 0.35],
    subplot_titles=("Pepper Mid Price", "Inventory (target = +60)", "Mark-to-Market PnL"),
)
fig_pp.add_trace(go.Scattergl(x=t_p, y=pp_mid_plot, mode="lines", line=dict(width=1, color="steelblue"), showlegend=False), row=1, col=1)
fig_pp.add_trace(go.Scattergl(x=t_p, y=pp_inv, mode="lines", line=dict(width=1.5, color="darkorange"), showlegend=False), row=2, col=1)
fig_pp.add_hline(y=PP_LONG_TGT * PEPPER_LIMIT, line_dash="dash", line_color="green", annotation_text="target +60", row=2, col=1)
fig_pp.add_hline(y=0, line_dash="dot", line_color="grey", row=2, col=1)
fig_pp.add_hline(y=PEPPER_LIMIT, line_dash="dash", line_color="red", annotation_text="+80 lim", row=2, col=1)
fig_pp.add_trace(go.Scattergl(x=t_p, y=pp_pnl, mode="lines", line=dict(width=1.5, color="green"), showlegend=False), row=3, col=1)
fig_pp.add_hline(y=0, line_dash="dot", line_color="grey", row=3, col=1)
fig_pp.update_layout(height=750, width=1200, title="PEPPER - pure_as_regime Inventory & PnL Simulation", template="plotly_white")
fig_pp.show()

# ── Summary stats ────────────────────────────────────────────────────────
print(f"ASH   final inv: {ash_inv[-1]:+d}   final PnL: {ash_pnl[-1]:,.2f}")
print(f"        avg |inv|: {np.mean(np.abs(ash_inv)):.1f}   max |inv|: {max(np.abs(ash_inv))}")
print(f"PEPPER final inv: {pp_inv[-1]:+d}   final PnL: {pp_pnl[-1]:,.2f}")
print(f"        avg inv:   {np.mean(pp_inv):.1f}   max inv: {max(pp_inv)}")


In [ ]:
# =============================================================================
# QUICK TESTS: does volatility regime logic materially change inventory/risk/PnL?
# Run after Cell 13 (uses ash_sim + simulate_ash from that cell)
# =============================================================================
import pandas as pd
import numpy as np
import plotly.graph_objects as go


def _run_ash_case(name, **overrides):
    # mutate globals used by simulate_ash in Cell 13, then restore
    keys = [
        "GAMMA_Q", "GAMMA_S", "STORM_LIM", "TAKE_REGIME_GATE",
        "SP_TH", "SP_CEIL", "BL_ALPHA"
    ]
    backup = {k: globals().get(k) for k in keys}
    try:
        for k, v in overrides.items():
            globals()[k] = v
        inv, pnl, phv, spr = simulate_ash(ash_sim)
        return {
            "case": name,
            "final_pnl": float(pnl[-1]),
            "avg_abs_inv": float(np.mean(np.abs(inv))),
            "max_abs_inv": float(np.max(np.abs(inv))),
            "pct_storm_ticks": float(np.mean(np.array(phv) > 0.2)),
            "avg_spread": float(np.mean(spr)),
            "pct_spread_floor": float(np.mean(np.array(spr) <= MIN_SP + 1e-9)),
            "inv_path": inv,
            "pnl_path": pnl,
        }
    finally:
        for k, v in backup.items():
            globals()[k] = v


cases = [
    _run_ash_case("baseline"),
    _run_ash_case(
        "no_regime_controls",
        GAMMA_S=GAMMA_Q,
        STORM_LIM=POSITION_LIMIT,
        TAKE_REGIME_GATE=1.1,
    ),
    _run_ash_case(
        "strong_regime_controls",
        GAMMA_S=0.35,
        STORM_LIM=20,
        TAKE_REGIME_GATE=0.05,
    ),
]

res_df = pd.DataFrame([{k: v for k, v in c.items() if not isinstance(v, list)} for c in cases])
res_df["delta_pnl_vs_baseline"] = res_df["final_pnl"] - res_df.loc[res_df["case"] == "baseline", "final_pnl"].iloc[0]
res_df["delta_avg_abs_inv_vs_baseline"] = res_df["avg_abs_inv"] - res_df.loc[res_df["case"] == "baseline", "avg_abs_inv"].iloc[0]

print("Quick A/B results (ASH):")
display(res_df.round(4).sort_values("final_pnl", ascending=False))

# Visual compare inventory and pnl paths
fig = go.Figure()
for c in cases:
    fig.add_trace(go.Scatter(y=c["inv_path"], mode="lines", name=f"inv | {c['case']}"))
fig.update_layout(title="ASH inventory path by regime setting", template="plotly_white", height=400, width=1100)
fig.show()

fig2 = go.Figure()
for c in cases:
    fig2.add_trace(go.Scatter(y=c["pnl_path"], mode="lines", name=f"pnl | {c['case']}"))
fig2.update_layout(title="ASH PnL path by regime setting", template="plotly_white", height=400, width=1100)
fig2.show()

print("Interpretation guide:")
print("- If no_regime_controls is near baseline, your current regime block has limited impact.")
print("- If strong_regime_controls reduces avg_abs_inv/max_abs_inv with small pnl hit, regime is mostly risk control.")
print("- If baseline beats both, current calibration is likely useful.")


In [ ]:
# =============================================================================
# Noise distribution diagnostics (ASH)
# Goal: characterize whether residual noise is Gaussian or heavy-tailed/jumpy.
# =============================================================================
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy import stats

# Pick the best available "noise" series from prior cells
noise = None
noise_name = None
if "resid_kalman" in globals():
    noise = np.asarray(resid_kalman, dtype=float)
    noise_name = "resid_kalman (cell 8)"
elif "fair_df" in globals() and "residual" in fair_df.columns:
    noise = pd.to_numeric(fair_df["residual"], errors="coerce").values
    noise_name = "fair_df['residual'] (cell 6)"
elif "dist_kalman" in globals():
    noise = np.asarray(dist_kalman, dtype=float)
    noise_name = "dist_kalman (cell 12)"
elif "lr_stat" in globals():
    noise = np.asarray(lr_stat, dtype=float)
    noise_name = "lr_stat (returns)"
else:
    raise RuntimeError("No residual/noise series found. Run cells 6/8/12 first.")

noise = noise[np.isfinite(noise)]
if noise.size < 50:
    raise RuntimeError(f"Noise series '{noise_name}' too short ({noise.size}).")

# Standardize for shape tests (location/scale free)
z = (noise - noise.mean()) / (noise.std(ddof=1) + 1e-12)

# Fit Normal and Student-t to standardized noise
mu_n, sd_n = stats.norm.fit(z)
df_t, loc_t, scale_t = stats.t.fit(z)

# Tests
jb_stat, jb_p = stats.jarque_bera(z)
nt_stat, nt_p = stats.normaltest(z)
sw_stat, sw_p = stats.shapiro(z[:5000]) if len(z) > 3 else (np.nan, np.nan)  # cap for speed
ad = stats.anderson(z, dist="norm")

print(f"Noise source: {noise_name}")
print(f"n = {len(z)}")
print(f"mean={z.mean():.4f}, std={z.std(ddof=1):.4f}, skew={stats.skew(z):.4f}, excess_kurt={stats.kurtosis(z):.4f}")
print(f"Jarque-Bera: stat={jb_stat:.3f}, p={jb_p:.3g}")
print(f"D'Agostino K^2: stat={nt_stat:.3f}, p={nt_p:.3g}")
print(f"Shapiro-Wilk (<=5000 pts): stat={sw_stat:.3f}, p={sw_p:.3g}")
print(f"Anderson-Darling(normal): stat={ad.statistic:.3f}, crit@5%={ad.critical_values[2]:.3f}")
print(f"Student-t fit: df={df_t:.2f}, loc={loc_t:.4f}, scale={scale_t:.4f}")

# Histogram + fitted PDFs
x = np.linspace(np.nanpercentile(z, 0.5), np.nanpercentile(z, 99.5), 500)
pdf_n = stats.norm.pdf(x, mu_n, sd_n)
pdf_t = stats.t.pdf(x, df_t, loc_t, scale_t)

fig = make_subplots(
    rows=1, cols=2, subplot_titles=("Histogram + fitted PDFs", "QQ plot vs Normal"),
    horizontal_spacing=0.12
)

fig.add_trace(
    go.Histogram(x=z, histnorm="probability density", nbinsx=120, opacity=0.55, name="noise z"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=x, y=pdf_n, mode="lines", line=dict(width=2), name="Normal fit"),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=x, y=pdf_t, mode="lines", line=dict(width=2), name="Student-t fit"),
    row=1, col=1
)

# QQ plot
theo_q = stats.norm.ppf((np.arange(1, len(z) + 1) - 0.5) / len(z))
emp_q = np.sort(z)
fig.add_trace(
    go.Scattergl(x=theo_q, y=emp_q, mode="markers", marker=dict(size=3, opacity=0.35), name="QQ points"),
    row=1, col=2
)
qmin = float(min(theo_q.min(), emp_q.min()))
qmax = float(max(theo_q.max(), emp_q.max()))
fig.add_trace(
    go.Scatter(x=[qmin, qmax], y=[qmin, qmax], mode="lines", line=dict(dash="dash"), name="45° line"),
    row=1, col=2
)

fig.update_layout(
    template="plotly_white",
    height=520,
    width=1300,
    title=f"Noise Distribution Diagnostics — {noise_name}"
)
fig.update_xaxes(title_text="z", row=1, col=1)
fig.update_yaxes(title_text="density", row=1, col=1)
fig.update_xaxes(title_text="Normal theoretical quantiles", row=1, col=2)
fig.update_yaxes(title_text="Empirical quantiles", row=1, col=2)
fig.show()

print("\nInterpretation:")
print("- If p-values are tiny and QQ tails bend away from 45°: noise is not Gaussian.")
print("- If Student-t fit hugs histogram tails better than Normal: heavy tails / jumpy noise.")
print("- Lower fitted t df (e.g., <10) implies fatter tails.")


In [ ]:
# =============================================================================
# Price regime check (3 regimes) on standardized deviation from 10k
# =============================================================================
import numpy as np
import pandas as pd
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

# Use existing variable if available, else build from ASH mids.
if "mid" in globals():
    mid_px = np.asarray(mid, dtype=float)
else:
    ash_tmp = full_prices[full_prices["product"] == "ASH_COATED_OSMIUM"].sort_values(["day", "timestamp"])
    mid_px = pd.to_numeric(ash_tmp["mid_price"], errors="coerce").replace(0.0, np.nan).dropna().values

z = (mid_px - 10_000.0)
z = (z - np.nanmean(z)) / (np.nanstd(z) + 1e-12)
z = pd.Series(z).replace([np.inf, -np.inf], np.nan).dropna()

MAX_Z_POINTS = 8000
z_fit = z.iloc[-MAX_Z_POINTS:] if len(z) > MAX_Z_POINTS else z
print(f"Fitting 3-state MarkovRegression on standardized price deviation, n={len(z_fit)}")

mod3 = MarkovRegression(z_fit.values, k_regimes=3, trend="c", switching_variance=True)
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    res3 = mod3.fit(search_reps=25)

print(res3.summary())

# Smoothed probabilities (n x 3)
sm3 = res3.smoothed_marginal_probabilities
sm3_arr = np.asarray(sm3.values) if hasattr(sm3, "values") else np.asarray(sm3)
if sm3_arr.ndim == 1:
    sm3_arr = sm3_arr.reshape(-1, 1)

# Build transition matrix for generic k-regime model.
P = res3.regime_transition if hasattr(res3, "regime_transition") else None
if P is not None:
    P = np.asarray(P)
    # statsmodels may return shape (k,k,1)
    if P.ndim == 3:
        P = P[:, :, 0]
    print("\nTransition matrix P (rows=from, cols=to):")
    print(np.round(P, 4))
    with np.errstate(divide="ignore", invalid="ignore"):
        durs = 1.0 / (1.0 - np.clip(np.diag(P), None, 0.999999))
    print("Expected regime durations (ticks):", np.round(durs, 2))
else:
    print("\nTransition matrix not exposed directly by this statsmodels version.")

# Regime labels by highest smoothed probability
regime_label3 = np.argmax(sm3_arr, axis=1)
t = np.arange(len(z_fit))

fig = make_subplots(
    rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.06,
    row_heights=[0.45, 0.30, 0.25],
    subplot_titles=("z(mid-10k) with regime coloring", "Smoothed regime probabilities", "Most-likely regime"),
)

fig.add_trace(
    go.Scattergl(
        x=t, y=z_fit.values, mode="markers",
        marker=dict(size=3, color=regime_label3, colorscale="Viridis", opacity=0.55),
        name="z(mid-10k)",
    ),
    row=1, col=1
)
fig.add_trace(
    go.Scatter(x=t, y=z_fit.values, mode="lines", line=dict(width=1, color="lightgrey"), name="z line"),
    row=1, col=1
)

for j in range(sm3_arr.shape[1]):
    fig.add_trace(
        go.Scatter(x=t, y=sm3_arr[:, j], mode="lines", line=dict(width=1.5), name=f"P(regime {j})"),
        row=2, col=1
    )

fig.add_trace(
    go.Scattergl(x=t, y=regime_label3, mode="markers", marker=dict(size=3), name="argmax regime"),
    row=3, col=1
)

fig.update_yaxes(title_text="z", row=1, col=1)
fig.update_yaxes(title_text="prob", row=2, col=1)
fig.update_yaxes(title_text="label", row=3, col=1)
fig.update_layout(height=850, width=1250, template="plotly_white", title="3-Regime Price State Check")
fig.show()


In [ ]:
# =============================================================================
# Per-day 3-state Markov regime on residual noise (mid - 10k)
# Fits independently on each day to compare regime widths (sigma2),
# means (const), transition matrices, and durations across days.
# =============================================================================
import numpy as np
import pandas as pd
import warnings
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression

FAIR = 10_000.0
K_REGIMES = 3
SEARCH_REPS = 25

ash_all = full_prices[full_prices["product"] == "ASH_COATED_OSMIUM"].copy()
ash_all = ash_all.sort_values(["day", "timestamp"]).reset_index(drop=True)
ash_all["mid"] = pd.to_numeric(ash_all["mid_price"], errors="coerce").replace(0.0, np.nan)
days = sorted(ash_all["day"].unique())
print(f"Days in data: {days}\n")

results = {}

for day in days:
    sub = ash_all[ash_all["day"] == day].dropna(subset=["mid"])
    residual = sub["mid"].values - FAIR
    resid_std = (residual - np.nanmean(residual)) / (np.nanstd(residual) + 1e-12)
    s = pd.Series(resid_std).replace([np.inf, -np.inf], np.nan).dropna()

    print(f"{'='*60}")
    print(f"DAY {day}: n={len(s)} ticks")
    print(f"{'='*60}")

    mod = MarkovRegression(s.values, k_regimes=K_REGIMES, trend="c", switching_variance=True)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        res = mod.fit(search_reps=SEARCH_REPS)

    print(res.summary())

    pv = res.params.values if hasattr(res.params, "values") else np.asarray(res.params)

    regime_info = []
    for k in range(K_REGIMES):
        const_val = pv[2 * k]
        sig2_val = pv[2 * k + 1]
        regime_info.append({"regime": k, "const": const_val, "sigma2": sig2_val,
                            "sigma": np.sqrt(abs(sig2_val))})

    regime_info.sort(key=lambda r: r["sigma2"])
    labels = ["calm", "normal", "stormy"]
    for i, r in enumerate(regime_info):
        r["label"] = labels[i]
        print(f"  Regime {r['regime']} ({r['label']}): const={r['const']:.4f}, "
              f"sigma2={r['sigma2']:.6f}, sigma={r['sigma']:.4f}")

    P = res.regime_transition if hasattr(res, "regime_transition") else None
    trans_matrix = None
    durations = None
    if P is not None:
        P = np.asarray(P)
        if P.ndim == 3:
            P = P[:, :, 0]
        trans_matrix = P
        print(f"\n  Transition matrix:")
        print(np.round(P, 4))
        with np.errstate(divide="ignore", invalid="ignore"):
            durations = 1.0 / (1.0 - np.clip(np.diag(P), None, 0.999999))
        print(f"  Expected durations: {np.round(durations, 2)}")

    sm = res.smoothed_marginal_probabilities
    sm_arr = np.asarray(sm.values) if hasattr(sm, "values") else np.asarray(sm)
    if sm_arr.ndim == 1:
        sm_arr = sm_arr.reshape(-1, 1)

    frac_in_regime = sm_arr.mean(axis=0)
    print(f"  Fraction of time in each regime: {np.round(frac_in_regime, 4)}")

    results[day] = {
        "n": len(s),
        "regimes": regime_info,
        "trans_matrix": trans_matrix,
        "durations": durations,
        "frac_time": frac_in_regime,
        "smoothed": sm_arr,
        "series": s.values,
        "res": res,
    }
    print()

# ── COMPARISON TABLE ──────────────────────────────────────────────────────
print("\n" + "="*80)
print("CROSS-DAY COMPARISON TABLE")
print("="*80)
header = f"{'Metric':<30s}"
for d in days:
    header += f"  {'Day '+str(d):>18s}"
print(header)
print("-" * (30 + 20 * len(days)))

for label_idx, label in enumerate(["calm", "normal", "stormy"]):
    for metric in ["const", "sigma2", "sigma"]:
        row = f"  {label}_{metric:<22s}"
        for d in days:
            ri = results[d]["regimes"]
            val = ri[label_idx][metric] if label_idx < len(ri) else float("nan")
            row += f"  {val:>18.6f}"
        print(row)

row_dur = f"{'Expected durations':<30s}"
for d in days:
    dur = results[d]["durations"]
    if dur is not None:
        row_dur += f"  {str(np.round(dur, 1)):>18s}"
    else:
        row_dur += f"  {'N/A':>18s}"
print(row_dur)

row_frac = f"{'Frac time [calm/norm/storm]':<30s}"
for d in days:
    fr = results[d]["frac_time"]
    sorted_fr = [results[d]["regimes"][i]["label"] + f":{fr[results[d]['regimes'][i]['regime']]:.2f}"
                 for i in range(K_REGIMES)]
    row_frac += f"  {' '.join(sorted_fr):>18s}"
print(row_frac)

# ── CONSISTENCY CHECK ─────────────────────────────────────────────────────
print("\n" + "="*80)
print("CONSISTENCY CHECK: are regime widths stable across days?")
print("="*80)
for label_idx, label in enumerate(["calm", "normal", "stormy"]):
    sig2s = [results[d]["regimes"][label_idx]["sigma2"] for d in days]
    mean_s2 = np.mean(sig2s)
    spread = max(sig2s) - min(sig2s)
    cv = np.std(sig2s) / (mean_s2 + 1e-12)
    print(f"  {label:8s} sigma2 across days: {[f'{v:.6f}' for v in sig2s]}  "
          f"CV={cv:.3f}  {'STABLE' if cv < 0.25 else 'DIFFERS'}")

# ── PER-DAY PLOTS ────────────────────────────────────────────────────────
fig = make_subplots(
    rows=len(days), cols=2,
    shared_xaxes=False, vertical_spacing=0.08, horizontal_spacing=0.08,
    subplot_titles=[item for d in days for item in
                    (f"Day {d}: residual with regime color",
                     f"Day {d}: smoothed regime probs")],
)

colors_map = {0: "blue", 1: "green", 2: "red"}
for row_i, d in enumerate(days, 1):
    r = results[d]
    s = r["series"]
    sm = r["smoothed"]
    regime_labels = np.argmax(sm, axis=1)
    t = np.arange(len(s))

    regime_order = [r["regimes"][i]["regime"] for i in range(K_REGIMES)]

    fig.add_trace(
        go.Scattergl(
            x=t, y=s, mode="markers",
            marker=dict(size=2, color=regime_labels,
                        colorscale=[[0, "royalblue"], [0.5, "limegreen"], [1, "crimson"]],
                        opacity=0.5),
            name=f"Day {d} residual", showlegend=False,
        ),
        row=row_i, col=1,
    )

    for j in range(sm.shape[1]):
        lbl = "?"
        for ri in r["regimes"]:
            if ri["regime"] == j:
                lbl = ri["label"]
        fig.add_trace(
            go.Scatter(x=t, y=sm[:, j], mode="lines", line=dict(width=1.5),
                       name=f"Day {d} P({lbl})", showlegend=(row_i == 1)),
            row=row_i, col=2,
        )

fig.update_layout(
    height=350 * len(days), width=1300, template="plotly_white",
    title="Per-Day 3-State Markov Regime on Residual Noise (mid - 10k)",
)
fig.show()

# ── OVERLAY: regime sigma2 bar chart ─────────────────────────────────────
bar_fig = go.Figure()
x_labels = ["calm", "normal", "stormy"]
for d in days:
    sig2s = [results[d]["regimes"][i]["sigma2"] for i in range(K_REGIMES)]
    bar_fig.add_trace(go.Bar(name=f"Day {d}", x=x_labels, y=sig2s))
bar_fig.update_layout(
    barmode="group", template="plotly_white",
    title="Regime sigma2 by day (lower = tighter noise)",
    yaxis_title="sigma2 (standardized residual)",
    height=400, width=700,
)
bar_fig.show()